# IBM App Connect Enterprise 12.0.x — Reference crawler

Run this notebook from your `ace-rag/` project root.

**Workflow**
1. Run setup and configuration.
2. Run discovery and inspect the topic list.
3. If coverage looks correct, set `EXTRACT_ARTICLES = True` in the configuration cell, rerun it, then run extraction and export.

This is a first-pass crawler; IBM Docs navigation markup may change. Validate discovered URLs and extracted text before merging the output into your curated ACE guide.


## 1. Install Playwright

In [ ]:
%pip install -q playwright
!playwright install chromium


## 2. Configuration

In [1]:
from pathlib import Path

START_URL = "https://www.ibm.com/docs/en/app-connect/12.0.x?topic=software-reference"

OUTPUT_DIR = Path("data/web_reference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DISCOVERED_TOPICS_PATH = OUTPUT_DIR / "discovered_topics.json"
RECORDS_PATH = OUTPUT_DIR / "reference_pages_content.jsonl"
COMBINED_TEXT_PATH = OUTPUT_DIR / "reference_combined.txt"
FAILED_URLS_PATH = OUTPUT_DIR / "failed_urls.txt"

REQUEST_DELAY_SECONDS = 1.0
NAVIGATION_TIMEOUT_MS = 45_000
HEADLESS = False

# Keep False for discovery-only. Change to True after checking discovered_topics.json.
EXTRACT_ARTICLES = True

print("Output directory:", OUTPUT_DIR.resolve())


Output directory: C:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\data\web_reference


## 3. Imports and helper functions

In [2]:
import asyncio
import json
import re
import sys
import threading
from datetime import datetime, timezone
from urllib.parse import parse_qs, urlencode, urlparse, urlunparse

from playwright.async_api import async_playwright


def run_playwright(async_fn, *args, **kwargs):
    """Run an async Playwright coroutine on a subprocess-capable event loop.

    Jupyter on Windows uses SelectorEventLoop, which cannot spawn Playwright's
    Node driver. That shows up as a bare NotImplementedError at launch.
    """
    box = {}

    def _runner():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["result"] = loop.run_until_complete(async_fn(*args, **kwargs))
        except BaseException as exc:
            box["error"] = exc
        finally:
            loop.close()

    thread = threading.Thread(target=_runner)
    thread.start()
    thread.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


def normalize_url(raw_url: str):
    """Keep only IBM App Connect 12.0.x documentation URLs with a topic ID."""
    if not raw_url:
        return None
    parsed = urlparse(raw_url)
    if parsed.netloc.lower() not in {"www.ibm.com", "ibm.com"}:
        return None
    if not parsed.path.startswith("/docs/en/app-connect/12.0.x"):
        return None
    topic = parse_qs(parsed.query).get("topic", [])
    if not topic or not topic[0]:
        return None
    return urlunparse((
        "https", "www.ibm.com", parsed.path, "", urlencode({"topic": topic[0]}), ""
    ))


def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(record, path=RECORDS_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\\n")


def load_completed_urls(path=RECORDS_PATH):
    completed = set()
    if not path.exists():
        return completed
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                if record.get("url"):
                    completed.add(record["url"])
            except (json.JSONDecodeError, AttributeError):
                pass
    return completed

print("Helpers ready.")


Helpers ready.


## 4. Find and expand the Reference navigation

In [ ]:
async def get_topic_id_from_url(url: str) -> str | None:
    if not url:
        return None
    parsed = urlparse(url)
    q = parse_qs(parsed.query).get("topic", [])
    return q[0] if q and q[0] else None


async def find_tree_root_for_topic(page, topic_id: str):
    # Try to locate the anchor for the specific topic ID anywhere in the nav/tree,
    # then return the nearest tree item container (so we only crawl that branch).
    candidates = [
        page.locator(f'a[href*="topic={topic_id}"]'),
        page.locator('a[href*="topic=software-reference"]') if topic_id == "software-reference" else None,
        page.get_by_role("link", name=re.compile(r"^Reference$", re.I)) if topic_id == "software-reference" else None,
    ]
    candidates = [c for c in candidates if c is not None]

    anchor = None
    for locator in candidates:
        try:
            for i in range(min(await locator.count(), 20)):
                candidate = locator.nth(i)
                if await candidate.is_visible():
                    anchor = candidate
                    break
        except Exception:
            continue
        if anchor is not None:
            break

    if anchor is None:
        # As a last resort, try to find any visible link with this topic ID
        locator = page.locator(f'a[href*="topic={topic_id}"]')
        if await locator.count():
            anchor = locator.first if await locator.first.is_visible() else None

    if anchor is None:
        raise RuntimeError(f"Could not locate navigation anchor for topic={topic_id}")

    # Prefer the nearest list/tree item so we only crawl this topic branch.
    for xpath in [
        "xpath=ancestor::li[1]",
        "xpath=ancestor::*[@role='treeitem'][1]",
        "xpath=ancestor::nav[1]",
        "xpath=ancestor::*[@role='navigation'][1]",
    ]:
        root = anchor.locator(xpath)
        if await root.count():
            return root

    return anchor


async def find_current_tree_root(page):
    topic_id = await get_topic_id_from_url(page.url)
    if not topic_id:
        # fallback to START_URL topic
        topic_id = await get_topic_id_from_url(normalize_url(START_URL) or START_URL)
    if not topic_id:
        raise ValueError("Could not determine current topic id from URL.")
    return await find_tree_root_for_topic(page, topic_id)


async def expand_tree(root, max_passes=40):
    for _ in range(max_passes):
        buttons = root.locator('button[aria-expanded="false"]')
        count = await buttons.count()
        if count == 0:
            break
        clicked = 0
        for i in range(count):
            try:
                button = buttons.nth(i)
                if await button.is_visible():
                    await button.click(timeout=1500)
                    clicked += 1
                    await asyncio.sleep(0.15)
            except Exception:
                pass
        if clicked == 0:
            break


async def collect_links_from_subtree(page, subtree_root):
    links = subtree_root.locator("a[href*='topic=']")
    found = {}
    try:
        count = await links.count()
    except Exception:
        count = 0

    for i in range(count):
        link = links.nth(i)
        try:
            href = await link.evaluate("(el) => el.href")
            title = re.sub(r"\s+", " ", (await link.inner_text()).strip())
        except Exception:
            continue
        url = normalize_url(href)
        if url and title:
            found.setdefault(url, {"title": title, "url": url})
    return list(found.values())


print("Recursive navigation helpers ready.")

## 5. Discover topics

In [ ]:
async def discover_topics_recursive():
    start_url = normalize_url(START_URL)
    if not start_url:
        raise ValueError("START_URL did not normalize to a valid IBM topic URL.")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()
        page.set_default_navigation_timeout(NAVIGATION_TIMEOUT_MS)

        discovered = {}  # url -> {title, url}
        visited = set()
        to_visit = [start_url]

        try:
            while to_visit:
                current = to_visit.pop()
                if current in visited:
                    continue
                visited.add(current)

                try:
                    response = await page.goto(current, wait_until="domcontentloaded")
                    await page.wait_for_timeout(1200)
                    if current == start_url:
                        print("Landed on:", page.url)
                        if response:
                            print("HTTP status:", response.status)

                    # Get this page's visible subtree in the left nav and expand it
                    try:
                        root = await find_current_tree_root(page)
                        await expand_tree(root)
                    except Exception as nav_exc:
                        # If we cannot find/expand the nav for this page, skip discovering children
                        print(f"Warning: could not expand nav for {current}: {nav_exc}")
                        root = None

                    # Optionally capture this page's title (from H1) so it's recorded too
                    page_title = ""
                    for selector in ["main h1", "article h1", "h1", "title"]:
                        try:
                            loc = page.locator(selector).first
                            if await loc.count():
                                page_title = (await loc.inner_text()).strip()
                                if page_title:
                                    break
                        except Exception:
                            pass

                    if page_title:
                        discovered.setdefault(current, {"title": re.sub(r"\s+", " ", page_title), "url": current})
                    else:
                        # If no H1, at least ensure the URL appears
                        discovered.setdefault(current, {"title": current, "url": current})

                    # Collect descendant links from this node's subtree and enqueue them
                    if root is not None:
                        children = await collect_links_from_subtree(page, root)
                        for item in children:
                            url = item["url"]
                            if url not in discovered:
                                discovered[url] = item
                            if url not in visited and url not in to_visit:
                                to_visit.append(url)

                    await asyncio.sleep(REQUEST_DELAY_SECONDS)
                except Exception as exc:
                    print("Discovery error at", current, "-", exc)
                    continue
        finally:
            await browser.close()

    topics = list(discovered.values())
    topics.sort(key=lambda item: item["title"].casefold())
    save_json(topics, DISCOVERED_TOPICS_PATH)
    print(f"Discovered {len(topics)} topics (recursive).")
    print("Saved:", DISCOVERED_TOPICS_PATH.resolve())
    return topics


# Run recursive discovery and print a sample
topics = run_playwright(discover_topics_recursive)
for i, topic in enumerate(topics[:100], start=1):
    print(f"{i:03}. {topic['title']} — {topic['url']}")
if len(topics) > 100:
    print(f"... and {len(topics) - 100} more.")

### Check the discovered list

Open `data/web_reference/discovered_topics.json` and verify that the expected Reference sections and child topics appear. If only the landing page appears, **do not run extraction yet**; the navigation selectors need adjusting for the current IBM Docs page.


## 6. Extract article text

In [3]:
async def extract_article(page, url):
    response = await page.goto(url, wait_until="domcontentloaded")
    await page.wait_for_timeout(1200)

    article = None
    for selector in ["main article", "article", "main", '[role="main"]']:
        locator = page.locator(selector)
        try:
            for i in range(min(await locator.count(), 3)):
                candidate = locator.nth(i)
                text = (await candidate.inner_text()).strip()
                if len(text) > 80:
                    article = candidate
                    break
        except Exception:
            pass
        if article is not None:
            break

    content = (
        (await article.inner_text()).strip()
        if article is not None
        else (await page.locator("body").inner_text()).strip()
    )

    title = ""
    for selector in ["main h1", "article h1", "h1"]:
        try:
            locator = page.locator(selector).first
            if await locator.count():
                title = (await locator.inner_text()).strip()
                if title:
                    break
        except Exception:
            pass
    if not title:
        title = (await page.title()).strip()

    return {
        "title": re.sub(r"\\s+", " ", title),
        "url": normalize_url(url) or url,
        "source": "IBM Documentation — App Connect Enterprise 12.0.x Reference",
        # This first version does not reconstruct the full nested TOC hierarchy.
        "parent_topic": "Reference",
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "http_status": response.status if response else None,
        "text": content,
    }


async def extract_all_topics():
    if not DISCOVERED_TOPICS_PATH.exists():
        raise FileNotFoundError("Run discovery first.")

    discovered = json.loads(DISCOVERED_TOPICS_PATH.read_text(encoding="utf-8"))
    completed = load_completed_urls()
    pending = [item for item in discovered if item["url"] not in completed]
    print(f"Discovered: {len(discovered)} | already extracted: {len(completed)} | remaining: {len(pending)}")

    failed = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        page = await browser.new_page()
        page.set_default_navigation_timeout(NAVIGATION_TIMEOUT_MS)
        try:
            for index, item in enumerate(pending, start=1):
                print(f"[{index}/{len(pending)}] {item['title']} — {item['url']}")
                try:
                    record = await extract_article(page, item["url"])
                    if len(record["text"]) < 80:
                        print("  Warning: very little text extracted.")
                    append_jsonl(record)
                except Exception as exc:
                    failed.append({
                        "url": item["url"],
                        "title": item.get("title", ""),
                        "error": str(exc),
                    })
                    print("  FAILED:", exc)
                await asyncio.sleep(REQUEST_DELAY_SECONDS)
        finally:
            await browser.close()

    if failed:
        FAILED_URLS_PATH.write_text(
            "\\n".join(json.dumps(x, ensure_ascii=False) for x in failed),
            encoding="utf-8",
        )
        print("Failures saved to:", FAILED_URLS_PATH.resolve())
    else:
        print("No extraction failures recorded.")
    print("JSONL:", RECORDS_PATH.resolve())
    return failed


if EXTRACT_ARTICLES:
    failed = run_playwright(extract_all_topics)
else:
    print("Skipped. Set EXTRACT_ARTICLES = True in the configuration cell, rerun it, then run this cell.")


Discovered: 1579 | already extracted: 0 | remaining: 1579
[1/1579] .NET reference — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=interfaces-net-reference
[2/1579] .NETCompute node — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=nodes-netcompute-node
[3/1579] .NETInput node — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=nodes-netinput-node
[4/1579] _MB superglobal variables — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=patterns-mb-superglobal-variables
[5/1579] Abend files — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=troubleshooting-abend-files
[6/1579] Abend in a user-defined extension — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=troubleshooting-abend-in-user-defined-extension
[7/1579] ABS and ABSVAL functions — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=functions-abs-absval
[8/1579] AbstractPropertyEditor — https://www.ibm.com/docs/en/app-connect/12.0.x?topic=package-abstractpropertyeditor
[9/1579] ACORD messaging standard 

## 7. Export pages to one combined text file

In [7]:
RECORDS_PATH = OUTPUT_DIR / "reference_pages_content_formated.jsonl"

In [8]:
def export_combined_text():
    if not RECORDS_PATH.exists():
        raise FileNotFoundError(f"No extracted records at {RECORDS_PATH}. Run extraction first.")

    sections = []
    with RECORDS_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            sections.append("\\n".join([
                "=" * 88,
                f"TITLE: {record.get('title', 'Untitled')}",
                f"URL: {record.get('url', '')}",
                f"SOURCE: {record.get('source', '')}",
                f"RETRIEVED_AT_UTC: {record.get('retrieved_at_utc', '')}",
                "=" * 88,
                "",
                record.get("text", "").strip(),
                "",
            ]))

    combined = "\\n".join(sections)
    COMBINED_TEXT_PATH.write_text(combined, encoding="utf-8")
    print(f"Exported {len(sections)} pages.")
    print("Combined text:", COMBINED_TEXT_PATH.resolve())
    print(f"Character count: {len(combined):,}")
    return combined


if RECORDS_PATH.exists():
    combined_text = export_combined_text()
    print(combined_text[:3000])
else:
    print("No JSONL records yet; run extraction first.")


Exported 1573 pages.
Combined text: C:\Users\ME36352\MyFolder\AI\Repos\AIRepo\aceRag\data\web_reference\reference_combined.txt
Character count: 5,482,753
========================================================================================\nTITLE: .NET reference\nURL: https://www.ibm.com/docs/en/app-connect/12.0.x?topic=interfaces-net-reference\nSOURCE: IBM Documentation — App Connect Enterprise 12.0.x Reference\nRETRIEVED_AT_UTC: 2026-10-29T08:07:33.100498+00:00\n========================================================================================\n\n.NET reference
Last Updated
: 2026-10-23

You can call .NET assemblies from .NETCompute nodes, and from nodes that support ESQL.

The .NET classes and methods are described in the .NET API documentation.

Parent topic:
Transformation interfaces\n\n========================================================================================\nTITLE: .NETCompute node\nURL: https://www.ibm.com/docs/en/app-connect/12.0.x?topic=nodes-netcomput

## Notes / limitations

- The navigation selectors may need adjustment if IBM changes its documentation UI.
- This initial version records `parent_topic = "Reference"` for every page; it does not preserve the full nested table-of-contents hierarchy.
- Check several records for navigation text, missing content, or empty pages before using them for RAG.
- Extraction resumes by skipping URLs already present in `reference_pages.jsonl`. Back up/remove that file to force a clean extraction.
- The notebook writes separate output files and does **not** modify your curated ACE guide automatically. Review and deduplicate before merging, preserving the source URLs.


In [9]:
import json

# Change these paths as needed
input_path = "data/web_reference/reference_pages_content_formated.jsonl"
output_path = "data/web_reference/reference_documentation.txt"

count = 0
with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8", newline="\n") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            # Skip malformed lines
            continue

        text = obj.get("text")
        if text is None:
            continue
        if not isinstance(text, str):
            text = str(text)

        # Ensure literal "\n" sequences become real newlines
        text = text.replace("\\n", "\n")

        fout.write(text)
        # Separate records by a newline if not already ending with one
        if not text.endswith("\n"):
            fout.write("\n")

        count += 1

print(f"Wrote {count} text entries to {output_path}")

Wrote 1573 text entries to data/web_reference/reference_documentation.txt


In [ ]:
import re

# Change these paths as needed
input_path = "data/web_reference/reference_documentation.txt"
output_path = "data/web_reference/reference_documentation_step1.txt"

with open(input_path, "r", encoding="utf-8") as f:
    text = f.read()

# Remove:
# - "Parent topic:"
# - everything after it up to the next colon (:)
# - that colon and the next 11 characters
pattern = re.compile(r"Last Updated.*?:.{11}", flags=re.DOTALL)
cleaned_text, num_removed = pattern.subn("", text)

with open(output_path, "w", encoding="utf-8", newline="\n") as f:
    f.write(cleaned_text)

print(f"Removed {num_removed} occurrence(s). Wrote cleaned text to {output_path}")

Removed 1150 occurrence(s). Wrote cleaned text to data/web_reference/reference_documentation_step1.txt
